# Scoring a coarse-grained protein

A coarse-grained protein model is one bead or a few per residue, and the
energy that holds it together is a sum of *knowledge-based* terms -- what
the PDB says residues of these types usually do at this distance -- and
*steric* terms -- what they physically cannot do.

**They are IMP scores.** A coarse-grained potential needs coordinates, a
residue lookup and a pair loop with a cutoff, and
IMP has all three -- particles in a `Model`, a
`Hierarchy` over them, and a `ClosePairContainer`.
So the terms compose with IMP's optimizers and scoring
functions like any other restraint, instead of being a second scoring
world beside them.


## A structure to score

T4 lysozyme, read as C-alphas -- one bead per residue.


In [ ]:
import numpy as np

import IMP
import IMP.atom
import IMP.container
import IMP.core
import IMP.bff

model = IMP.Model()
hierarchy = IMP.atom.read_pdb(
    IMP.bff.get_example_path('structure/T4L/3GUN.pdb'), model,
    IMP.atom.CAlphaPDBSelector())
cas = [a.get_particle_index() for a in IMP.atom.get_leaves(hierarchy)]
len(cas)


## The Gō model

A well per residue pair, centred on the distance *this* structure has:
pairs inside the cutoff are native and get the full depth, the rest get a
fraction of it. The energy is a truncated Lennard-Jones, so a fold at its
own native contacts sits at the floor and every departure costs.

The contacts are taken when the restraint is built. `set_native_contacts`
retakes them, which is how a different conformer becomes the reference.


In [ ]:
go = IMP.bff.GoRestraint(model, cas, epsilon=1.0, cutoff=6.5,
                         nn_e_factor=0.1)
print(go.get_n_native(), 'native contacts of',
      go.get_contacts().get_n_contacts())
go.unprotected_evaluate(None)


In [ ]:
# the wells, as numpy: one depth and one minimum per pair
go.get_contacts().well_depths[:5], go.get_contacts().well_minima[:5]


## Sterics

`create_clash_restraint` is IMP's own `SoftSpherePairScore` over a
`ClosePairContainer`, with the bonded pairs filtered out by
`IMP.atom.StereochemistryPairFilter` when the hierarchy carries bonds.

The ported potential is $((\sigma - r)/t)^2$ and IMP's soft sphere is
$\tfrac{1}{2}k(\sigma - r)^2$, so the builder uses $k = 2/t^2$ and the
two are the same term. The reference implementation skipped pairs closer than a
`covalent_radius`, which was a distance guess at *these are bonded*; IMP
has the bond graph and can say so exactly.


In [ ]:
for a in IMP.atom.get_leaves(hierarchy):
    IMP.core.XYZR(a).set_radius(2.5)

clash = IMP.bff.create_clash_restraint(hierarchy, clash_tolerance=2.0)
IMP.core.RestraintsScoringFunction([clash]).evaluate(False)


## Solvent exposure

Shrake-Rupley with one sphere per residue: the coarse-grained form of the
accessible surface, composed from `sphere_points` and
`solvent_accessible_surface_area`, which this module already had for
atoms.


In [ ]:
IMP.bff.residue_solvent_accessible_surface(
    hierarchy, IMP.atom.AT_CA, n_sphere=590, probe=1.0, radius=2.5)


## Residue-typed contact potentials

Miyazawa-Jernigan and the UNRES side-chain centroid potential are the
same shape: a table indexed by the two residue types and, for UNRES, by
the distance. That shape is `IMP.core.StatisticalPairScore`, which is
what `IMP.atom.DopePairScore` is built on -- so the potential is *data*
and there is no loop to write.

`add_residue_type_score_data` puts the residue type on one atom per
residue, the way `IMP.atom.add_dope_score_data` does for DOPE. A residue
without that atom -- a glycine has no C-beta -- takes no part.


In [ ]:
# an all-atom copy, because the C-beta is what Miyazawa-Jernigan scores
m2 = IMP.Model()
all_atom = IMP.atom.read_pdb(
    IMP.bff.get_example_path('structure/T4L/3GUN.pdb'), m2,
    IMP.atom.NonWaterNonHydrogenPDBSelector())
typed = IMP.bff.add_residue_type_score_data(all_atom, IMP.atom.AT_CB)
key = IMP.bff.get_residue_type_key()
len(typed), typed[0].get_value(key)


Every table this module ships is in **one container**,
`data/potentials.pto` -- the same shape as `dyes.drot.pto`. The
default constructors read it, so a caller says what it wants
scored and not where the numbers live.

One file rather than four because a potential is not one table:
the UNRES term needs its grid *and* the residue order that
indexes it, the Ramachandran map needs to say which channel is
glycine, and a reader that has to find four files in agreement
with each other will one day find three.


In [ ]:
IMP.bff.potential_table_names()


In [ ]:
mj = IMP.container.PairsRestraint(
    IMP.bff.MiyazawaJerniganPairScore(),
    IMP.container.ClosePairContainer(
        IMP.container.ListSingletonContainer(
            m2, [p.get_index() for p in typed]), 6.5, 0.0))
IMP.core.RestraintsScoringFunction([mj]).evaluate(False)


The two contact potentials go into the container as **text in
IMP's PMF format**, which `IMP.core.StatisticalPairScore` reads
directly -- so the potential *is* the file and there is no loop
in this module that touches it. The file names residues and
IMP maps names to its own indices.


In [ ]:
t = IMP.bff.read_potential_table('mj')
print(t.kind)
print(t.text.split(chr(10))[0])          # bin width, type count
print(t.text.split(chr(10))[1])          # one line per type pair


The manifest says what each table is and **what the conversion
did to it** -- the part four loose `.npy` files cannot carry.
None of these were copies: the UNRES table had six NaNs, the
hydrogen-bond lookup diverges to 7e33 below 1.3 Å, and two of
the Ramachandran map's five channels are coordinate grids
rather than maps.


In [ ]:
import json
manifest = json.loads(IMP.bff.read_potential_manifest())
for name, entry in manifest['tables'].items():
    print(name, '--', entry['note'])


## Hydrogen bonds

A bond is an amide H within a few Angstrom of a carbonyl O, and it is
scored by four distances at once -- O-H, O-N, C-H, C-N -- out of a
four-channel lookup. Both directions of a residue pair are tested, so an
antiparallel pair can contribute twice, and each channel can be switched
off.

The C-alpha cutoff **is applied**. The kernel this was ported from
compared a plain C-alpha distance against the *square* of its cutoff, so
its prefilter passed everything and the 8 Å it advertised was never in
force; see `okf/validation/hbond_ca_cutoff.md`.


In [ ]:
hb = IMP.bff.read_potential_table('hbond')
r = IMP.bff.HydrogenBondRestraint(m2, all_atom, hb.table,
                                  hb.shape[1],
                                  cutoff_ca=8.0, cutoff_h=3.0)
# a crystal structure read without hydrogens has no amide H, so
# there is nothing to donate and the term is zero
r.unprotected_evaluate(None), r.get_n_hbonds()


## Backbone dihedrals

`RamachandranRestraint` reads $(\phi, \psi)$ per residue out of a grid
as $-\log(P/P_{max})$, so a uniform map scores zero and an empty cell
costs a penalty. The first residue has no $\phi$ and the last no $\psi$;
both come back as NaN and are skipped.

ChiSurf's class of this name returned 0.0 and logged a warning -- the C
function it called had been lost.


In [ ]:
ra = IMP.bff.read_potential_table('ramachandran')
rama = IMP.bff.RamachandranRestraint(m2, all_atom, ra.table,
                                     ra.shape[0], ra.shape[1])
phi, psi = rama.phi, rama.psi
print(np.degrees(phi[1:4]), np.degrees(psi[1:4]))
rama.unprotected_evaluate(None)


## Putting them together

Every term is an `IMP.Restraint` or an `IMP.PairScore`, so a scoring
function is the ordinary one and so is the optimizer.


In [ ]:
sf = IMP.core.RestraintsScoringFunction([go, clash])
sf.evaluate(False)


## What is not here

* The radius of gyration is `IMP.atom.get_radius_of_gyration`, which IMP
  has already.
* A harmonic tether to a reference conformer's C-alpha internal
  coordinates is `create_ca_internal_restraints`, which assembles
  `IMP.core.Harmonic` distance, angle and dihedral restraints -- IMP's
  own, not a fourth copy of the arithmetic.
* `IMP.atom.CAAngleRestraint` and `IMP.atom.CADihedralRestraint` are
  IMP's *statistical* C-alpha terms, a score per angle bin. They answer a
  different question from the harmonic tether and both are worth having.
